# Od klávesnice do neuronu · lekce na GPU zdarma

Runtime → Change runtime type → **T4 GPU**, pak Runtime → Run all. Zhruba deset minut. Každá buňka je jedna lekce z přednášky; čísla, která vytiskne, jsou čísla ze slidů (teplota 0, na GPU by měla sedět do poslední číslice).

In [ ]:
!git clone -q https://github.com/moudrkat/od-klavesnice-do-neuronu.git
%cd od-klavesnice-do-neuronu/dovnitr
!pip install -q transformers scikit-learn

## Lekce 2 · Jak malý je drobek
28 vrstev, proud 1 536 čísel, 8 960 neuronů na vrstvu, 12 hlav, 151 936 tokenů, 1,54 miliardy parametrů. (HumanEval, 164 úloh, je volitelná buňka na konci.)

In [ ]:
!python model_facts.py

## Lekce 3 · Co jde dovnitř
Do modelu jde 87 tokenů. 23 jsou tvoje.

In [ ]:
!python token_strips.py
!python attn_split.py

## Lekce 4 · Vrstvy
Prvních 8 z 1 536 čísel na pozici `return`, vrstva po vrstvě, a co by model řekl, kdyby skončil tady.

In [ ]:
!python stream_walk.py

## Lekce 5 · Vrstva 20 a hýbání
Hlava 20·6 se dívá na `list`. Neuron 20·3545 svítí na `return`. Pak s nimi hýbeš a kód se mění.

In [ ]:
!python heads_matter.py
!python layer20.py
!python play20_grid.py

## A teď na *tvém* promptu
Napiš zadání na jednu funkci v Pythonu, která něco vrací (anglicky, model je na angličtinu učený). Builder 1,5B k ní napíše kód, uloží se do `orch_lock.json` a lekce 3 až 5 proběhnou znovu na tvém promptu.

In [ ]:
request = "Write a Python function longest_word(words): return the longest string in the list."  #@param {type:"string"}
import json, re, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
L = json.load(open("orch_lock.json"))
I = "Qwen/Qwen2.5-Coder-1.5B-Instruct"; t = AutoTokenizer.from_pretrained(I); m = AutoModelForCausalLM.from_pretrained(I, dtype=torch.bfloat16).to("cuda").eval()
tmpl = t.apply_chat_template([{"role":"system","content":L["builder_sys"]},{"role":"user","content":request}], tokenize=False, add_generation_prompt=True)
ids = t(tmpl, return_tensors="pt").input_ids.to("cuda")
with torch.no_grad(): g = m.generate(ids, max_new_tokens=160, do_sample=False, pad_token_id=t.eos_token_id)
code = t.decode(g[0, ids.shape[1]:], skip_special_tokens=True).strip()
print(code)
body = re.search(r"```(?:python)?\s*(.*?)```", code, re.S); body = body.group(1) if body else code
assert "return" in body, "lekce se dívají na slovo `return`; chtěj funkci, která něco vrací"
L["refiner_out"] = request; L["builder_out"] = code
json.dump(L, open("orch_lock.json","w"), ensure_ascii=False, indent=1)
del m; torch.cuda.empty_cache(); print("\nV orch_lock.json je teď tvůj prompt. Pusť tři buňky pod tím.")

In [ ]:
!python token_strips.py
!python attn_split.py

In [ ]:
!python stream_walk.py

In [ ]:
!python play20_grid.py

Stejný model, tvůj prompt: kolik tokenů bylo tvých, ve které vrstvě vyhrálo tvoje slovo a jestli neuron 20·3545 ještě něco dělá. Když ne, je to ta poctivá část přednášky: neurony nejsou stabilní adresy.

## Lekce 6 · Přičti směr, dostaneš JavaScript

In [ ]:
%cd ../scripts
!python lang_retest.py

## Lekce 7 · Probe
Požadavek na TOR a C2 bez jediného zlého slova. Probe (jeden směr ve vrstvě 14, z repa hidden-intent) ho označí; skóre po slovech ukáže, kde se to rozsvítí.

In [ ]:
!curl -sL https://raw.githubusercontent.com/moudrkat/hidden-intent/main/results/probe.json -o intent_probe.json && ls -la intent_probe.json
!python pertoken.py
!python eval_rig.py

## Lekce 1 · Parta (volitelné, stáhne i model 4B, ~8 GB)
Refiner (4B) → builder (1.5B) → reviewer (4B) na tvém přání.

In [ ]:
!python orchestrate.py '["Make me a to-do app so I can see how many tasks are done"]'

Každý skript zapsal JSON vedle sebe. Otevři je. Pak změň prompt v `orch_lock.json` a pusť lekci znovu: to je celá metoda.

## Lekce 2, volitelné · HumanEval, všech 164 úloh
Asi deset minut na T4. Výsledek: 110 ze 164.

In [ ]:
%cd ../dovnitr
!python humaneval_run.py